<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB13_Convolutional_Neural_Networks_Underwater_Hull_Images_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB13 · Clase 13 — Redes neuronales convolucionales sobre imágenes reales de inspección de casco submarina**

## Bloque 3: IA — Deep Learning (continuación)

`NB11`/`NB12` entrenaron redes con datos tabulares — el mismo tipo de características numéricas diseñadas a mano que usa el ML clásico. Esta clase es donde el Deep Learning demuestra su valor real: **imágenes en bruto**, donde diseñar a mano "las características correctas" sería mucho más difícil que para las bandas de frecuencia del sonar o los coeficientes de geometría del casco. Usamos el **[dataset LIACi](https://liaci.sintef.cloud)** real (SINTEF Ocean, Noruega): 1.893 imágenes reales de inspección submarina de casco de buque, anotadas a nivel de píxel en 10 categorías (crecimiento biológico marino, corrosión, ánodos, hélices, y más) — datos genuinamente navales y genuinamente no estructurados.

**Tarea de hoy**: un **clasificador binario de imágenes** — ¿muestra una imagen submarina dada una condición concreta (elegiremos una clase del dataset, p. ej. crecimiento biológico) o no? Es una tarea deliberadamente más sencilla que la segmentación completa a nivel de píxel, ajustada para encajar en una primera clase práctica de CNN; el problema completo de segmentación es real, más difícil, y se deja para más adelante.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar por qué un MLP normal encaja mal con datos de imagen en bruto, y qué calcula realmente una operación de convolución.
- Explicar el papel de las capas de pooling en una CNN, y cómo apilar bloques conv+pool construye desde los píxeles hasta características de alto nivel.
- Cargar y preparar datos de imagen reales para una CNN de PyTorch (redimensionado, normalización, construcción de etiquetas a partir de los propios datos).
- Construir, entrenar y evaluar un clasificador CNN pequeño de principio a fin, con el mismo rigor (train/val/test, dropout, evaluación honesta) que `NB11`/`NB12`.
- Visualizar qué detectan realmente las primeras capas de una CNN entrenada en una imagen real.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | ¿Por qué CNN para imágenes? Los límites de un MLP normal | 10 min | Teoría |
| 3 | La operación de convolución: kernels y mapas de características | 15 min | Teoría + Práctica |
| 4 | Pooling y arquitectura de una CNN, de principio a fin | 10 min | Teoría |
| 5 | Cargar y explorar el dataset real | 15 min | Práctica |
| 6 | Construir etiquetas de clasificación a partir de los propios datos | 15 min | Práctica |
| 7 | Preparar los tensores de imagen y una partición train/val/test | 10 min | Práctica |
| 8 | Práctica: construir y entrenar un clasificador CNN | 20 min | Práctica |
| 9 | Evaluar la CNN y visualizar qué ha aprendido | 15 min | Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una guía aproximada, no un guion cerrado — no hay descansos programados. Si terminamos todo con tiempo de sobra, la clase acaba antes; puede pasar y no hay problema.

> **Antes de empezar**: esta clase descarga un dataset real de ~1 GB y entrena con imágenes reales, ambas cosas más lentas que los datos tabulares de `NB11`/`NB12`. Si hay una GPU disponible, cámbiate a ella ahora (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`) — el entrenamiento irá notablemente más rápido.

---

## 1. Repaso: dónde estamos

- **`NB11`**: teoría del perceptrón → MLP, un primer clasificador real de PyTorch con datos tabulares de sonar.
- **`NB12`**: entrenar redes profundas correctamente — monitorización con validación, dropout, parada temprana, optimizadores.
- **`NB13`** (hoy): la arquitectura construida específicamente para imágenes — Redes Neuronales Convolucionales — sobre fotos reales de inspección submarina.

---

## 2. ¿Por qué CNN para imágenes? Los límites de un MLP normal

Nada nos impide meter una imagen en bruto en la red estilo `SonarMLP` de `NB11`: aplanar una imagen RGB de 96×96 en un vector de 96×96×3 = 27.648 números, y pasárselo a `nn.Linear`. Dos problemas reales con eso:

- **Explosión de parámetros**: una primera capa oculta de solo 64 unidades sobre esa entrada necesitaría más de 1,7 millones de pesos — para una sola capa, con imágenes pequeñas. Las fotos reales son mucho más grandes.
- **Ninguna noción de *dónde***: un vector aplanado trata el píxel (0,0) y el píxel (50,50) como números sin relación. No tiene ningún concepto incorporado de que los píxeles cercanos están relacionados, ni de que un patrón (un borde, una textura) significa lo mismo tanto si aparece en la esquina superior izquierda como en la inferior derecha de la imagen.

Una **Red Neuronal Convolucional (CNN)** arregla ambas cosas: desliza filtros pequeños y compartidos por la imagen (muchos menos parámetros, porque el mismo filtro se reutiliza en todas partes), y por construcción mira *vecindarios locales* de píxeles — `construyendo desde bordes, a texturas, a objetos completos, de forma totalmente automática`. Este es precisamente el caso de "datos en bruto, no estructurados" señalado ya en `NB11` §2 como el escenario donde el aprendizaje automático de características del deep learning realmente merece la pena.

> **Para saber más**: [Red neuronal convolucional (Wikipedia)](https://en.wikipedia.org/wiki/Convolutional_neural_network).

---

## 3. La operación de convolución: kernels y mapas de características

Un **kernel** (o filtro) es una matriz pequeña — digamos 3×3 — de pesos aprendibles. La convolución lo desliza por la imagen, y en cada posición multiplica el kernel elemento a elemento con los píxeles que tiene debajo, suma el resultado, y escribe ese único número en un **mapa de características** de salida. Kernels distintos detectan cosas distintas (bordes, esquinas, texturas); una CNN *aprende* los valores del kernel durante el entrenamiento, exactamente igual que aprende los pesos en el perceptrón de `NB11` — `un kernel no es más que un pequeño conjunto de pesos compartido`.

Calculemos uno a mano, sobre un ejemplo sintético diminuto — un borde vertical (mitad izquierda oscura, mitad derecha clara) y un kernel clásico detector de bordes verticales:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

input_grid = np.array([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
])
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
])

def convolve2d_valid(img, k):
    kh, kw = k.shape
    h, w = img.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i + kh, j:j + kw] * k)
    return out

output_grid = convolve2d_valid(input_grid, kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(input_grid, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Input (6x6) + one 3x3 kernel window")
for (i, j), val in np.ndenumerate(input_grid):
    axes[0].text(j, i, int(val), ha="center", va="center", color="red", fontsize=9)
axes[0].add_patch(patches.Rectangle((1.5, -0.5), 3, 3, linewidth=2, edgecolor="lime", facecolor="none"))
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].imshow(output_grid, cmap="viridis")
axes[1].set_title("Output feature map (4x4)")
for (i, j), val in np.ndenumerate(output_grid):
    axes[1].text(j, i, int(val), ha="center", va="center", color="white", fontsize=9)
axes[1].add_patch(patches.Rectangle((1.5, -0.5), 1, 1, linewidth=2, edgecolor="lime", facecolor="none"))
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.show()

La ventana 3×3 resaltada en la entrada produce exactamente el único valor resaltado en el mapa de características de salida — esa es una posición del deslizamiento. Fíjate en que la salida es grande donde la ventana del kernel cruza el límite oscuro/claro (¡un borde!) y casi cero en las zonas planas — este kernel concreto es, literalmente, un detector de bordes. En una CNN real nunca diseñamos los kernels a mano así; **la red aprende los valores del kernel que minimizan la pérdida**, que pueden detectar bordes, colores, texturas, o nada interpretable para un humano.

> **Para saber más**: [Kernel (procesado de imagen) (Wikipedia)](https://en.wikipedia.org/wiki/Kernel_%28image_processing%29) · [documentación de `torch.nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).

**Pruébalo tú mismo**: un kernel solo responde al patrón para el que está construido. Prueba un kernel detector de bordes *horizontales* (filas de +1/-1 en vez de columnas) sobre la misma entrada con rayas verticales — como no hay ningún borde horizontal en ella, ¿qué esperarías que pareciera la salida?

In [ ]:
horizontal_kernel = np.array([
    [1, 1, 1],
    [0, 0, 0],
    [-1, -1, -1],
])
horizontal_output = convolve2d_valid(input_grid, horizontal_kernel)
print("Horizontal-edge kernel output on the same (vertically-striped) input:")
print(horizontal_output)


---

## 4. Pooling y arquitectura de una CNN, de principio a fin

Una **capa de pooling** reduce un mapa de características resumiendo pequeñas regiones — **max pooling** (la opción habitual por defecto) conserva solo el valor más grande de cada ventana pequeña (p. ej. 2×2), reduciendo a la mitad ambas dimensiones. Dos ventajas: `hace la red más tolerante a que una característica se desplace uno o dos píxeles` (una pequeña traslación normalmente no cambia cuál era el valor máximo), y mantiene manejable el número de valores a medida que apilamos más capas.

Una CNN típica repite **Convolución → activación (ReLU) → Pooling** varias veces, cada bloque trabajando sobre una versión más gruesa y abstracta de la imagen, y después **aplana** (`flatten`) los mapas de características finales en un vector y los pasa a una pequeña "cabeza" MLP (exactamente las capas totalmente conectadas de `NB11`) que hace la predicción final:

| Etapa | Qué hace | Analogía con `NB11`/`NB12` |
|---|---|---|
| Capas convolucionales | Aprenden patrones locales (bordes → texturas → partes) | Las "características" que un modelo clásico necesitaría diseñadas a mano |
| ReLU | No linealidad entre capas | Mismo papel que en el MLP |
| Pooling | Reduce y resume | — nueva en las CNN |
| Aplanado + capas lineales | Combinan las características aprendidas en una predicción final | Idéntico a la cabeza `SonarMLP` de `NB11` |

Todo lo de `NB12` (dropout, parada temprana, Adam) sigue aplicándose sin cambios en cuanto llegamos a la cabeza totalmente conectada — una CNN es un extractor de características acoplado al mismo tipo de red que ya sabemos entrenar correctamente.

---

## 5. Cargar y explorar el dataset real

El **dataset LIACi** lo publica directamente SINTEF Ocean para uso en investigación. La descarga pesa aproximadamente 1 GB — esta celda puede tardar unos minutos:

In [ ]:
!wget -q -O liaci_data.zip https://liaci.sintef.cloud/download_data/data.zip
!unzip -oq liaci_data.zip
!ls

La inspección submarina de casco se hace normalmente de dos formas: enviar a un buceador a inspeccionar visualmente el casco, o desplegar un vehículo operado remotamente (ROV) con cámara. Ambas son caras, dependen del tiempo, y — para las inspecciones con buceador — genuinamente peligrosas; como resultado, los buques reales se inspeccionan con mucho menos frecuencia de la deseable, y mucho depende de que un experto humano identifique correctamente cada condición relevante en las imágenes que se capturen. Esta es precisamente el tipo de tarea — reconocimiento visual de patrones sobre imágenes reales, desordenadas y no estructuradas, a una escala y consistencia que ningún equipo de inspección pequeño podría igualar a mano — que motivó la entrada del Deep Learning en este curso, ya en `NB11` §2. Un modelo que señale con fiabilidad "aquí hay crecimiento biológico visible" a partir de una foto no sustituye al inspector, pero puede filtrar miles de fotogramas de una sola inspección con ROV hasta reducirlos al puñado que realmente necesita un ojo experto.

El archivo extrae una carpeta `images/` (las fotos en bruto) y una carpeta `masks/` (una subcarpeta por clase anotada, con un fichero de máscara del mismo nombre por cada imagen donde esa clase es visible). Veamos una imagen real:

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

base_dir = "LIACi_dataset_pretty"
imgs_path = os.path.join(base_dir, "images")
masks_path = os.path.join(base_dir, "masks")

image_files = sorted(os.listdir(imgs_path))
print("Total images:", len(image_files))

sample_img = Image.open(os.path.join(imgs_path, image_files[0]))
plt.imshow(sample_img)
plt.title(f"Example: {image_files[0]}")
plt.axis("off")
plt.show()

Y las clases de condición anotadas disponibles como máscaras (excluyendo dos carpetas auxiliares — `saliency` y `segmentation` — que contienen datos derivados en vez de una única clase semántica):

In [ ]:
mask_classes = sorted(f for f in os.listdir(masks_path) if f not in {"saliency", "segmentation"})
print(mask_classes)

---

## 6. Construir etiquetas de clasificación a partir de los propios datos

No hay ningún fichero de etiquetas "sí/no" ya preparado — pero podemos construir uno a partir de las máscaras: para una clase elegida, una imagen recibe la etiqueta **1** si su máscara para esa clase contiene algún píxel distinto de cero (la condición es visible en algún punto de la foto), y **0** en caso contrario.

Vamos a apuntar concretamente al **crecimiento biológico marino** ([biofouling](https://en.wikipedia.org/wiki/Biofouling)), y merece la pena detenerse en por qué esta condición, de las 11 disponibles, es `una elección genuinamente buena para un primer clasificador real — no solo visualmente distintiva, sino operativamente importante`: el biofouling en un casco aumenta la resistencia lo suficiente como para reducir la velocidad de un buque hasta un 10%, lo que puede exigir hasta un **40% más de combustible** para compensarlo, y solo la Marina de EE. UU. estima que le cuesta alrededor de **1.000 millones de dólares al año** en combustible extra, mantenimiento y medidas de control. Si esa cifra suena familiar, es porque lo es — es el mismo efecto físico detrás de las columnas reales `fuel_consumption`/`engine_efficiency` que modelamos en `NB07` y agrupamos en `NB09`, solo que aquí observado desde una cámara en vez de un registro de combustible. Un modelo que detecte el crecimiento con fiabilidad y a tiempo es, en un sentido muy directo, una herramienta para mantener más bajas las cifras de esos notebooks anteriores.

Elegiremos el nombre de clase que contenga "growth" en el archivo, sin fijar a mano la grafía exacta usada:

In [ ]:
target_class = next(c for c in mask_classes if "growth" in c.lower())
print("Target class for our binary classifier:", target_class)

Calcula la etiqueta para cada imagen:

In [ ]:
import numpy as np

def label_for(fname, cls):
    # Masks are stored as .bmp, regardless of the source image's extension
    # (here, .jpg) -- matching on the exact filename would silently fail
    # every lookup, so we match on the filename stem instead.
    stem = os.path.splitext(fname)[0]
    mask_file = os.path.join(masks_path, cls, stem + ".bmp")
    if not os.path.exists(mask_file):
        return 0
    mask = np.array(Image.open(mask_file).convert("L"))
    return int(mask.max() > 0)

all_labels = {f: label_for(f, target_class) for f in image_files}
positive = sum(all_labels.values())
print(f"{positive} / {len(all_labels)} images show '{target_class}'")

**Lee tu propio resultado**: ¿está la partición razonablemente equilibrada, o sesgada hacia una clase? Un dataset real construido así a menudo *no* está perfectamente equilibrado — la severidad del biofouling varía mucho según la zona del casco, el tiempo desde la última limpieza y el historial del buque, así que sería poco realista esperar una partición limpia 50/50. Esto importa por dos razones concretas, no solo como advertencia abstracta: primero, el punto de `NB07` de que la accuracy sola es una mala métrica para problemas desequilibrados sigue aplicando (un clasificador que siempre predice "Ausente" podría seguir obteniendo una accuracy alta si la condición es rara en los datos); segundo, cuando en la Parte 7 reduzcamos la muestra a un conjunto de entrenamiento manejable, una submuestra *puramente aleatoria* de un dataset ya desequilibrado puede acabar con apenas ejemplos de la clase minoritaria — algo que conviene diseñar deliberadamente, no dejar a la suerte.

Ver el desequilibrio como un diagrama de barras hace concreta la advertencia de la Parte 6:

In [ ]:
counts = [len(all_labels) - positive, positive]
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Absent", "Present"], counts, color=["steelblue", "darkorange"])
ax.set_ylabel("Number of images")
ax.set_title(f"Class balance for '{target_class}'")
plt.show()


---

## 7. Preparar los tensores de imagen y una partición train/val/test

Trabajar con las 1.893 imágenes a resolución completa sería lento para una clase en directo, así que tomamos una submuestra, redimensionamos cada imagen a un tamaño pequeño y consistente, y normalizamos los valores de píxel a 0–1. Dada la advertencia de la Parte 6, submuestreamos **de forma deliberada, no puramente aleatoria**: tomar por separado de las listas de imágenes positivas y negativas garantiza que ambas clases estén representadas de forma significativa en nuestros datos de entrenamiento, sin importar cuán desequilibrado esté el dataset completo. Es una técnica legítima y habitual — distinta de la fuga de datos, ya que solo estamos eligiendo *qué imágenes reales y sin modificar* incluir, no fabricando ni duplicando ninguna:

In [ ]:
import random

random.seed(42)

positive_files = [f for f in image_files if all_labels[f] == 1]
negative_files = [f for f in image_files if all_labels[f] == 0]
print(f"Available in the full dataset: {len(positive_files)} positive, {len(negative_files)} negative")

n_per_class = min(len(positive_files), len(negative_files), 200)
sample_files = random.sample(positive_files, n_per_class) + random.sample(negative_files, n_per_class)
random.shuffle(sample_files)

IMG_SIZE = 96

def load_image(fname, size=IMG_SIZE):
    img = Image.open(os.path.join(imgs_path, fname)).convert("RGB").resize((size, size))
    return np.array(img, dtype=np.float32) / 255.0

images = np.stack([load_image(f) for f in sample_files])
labels = np.array([all_labels[f] for f in sample_files])

print(images.shape, labels.shape)
print("Class balance in our sample:", np.bincount(labels))

Convierte a tensores de PyTorch — fíjate en el reordenado de dimensiones: PyTorch espera las imágenes como `(batch, channels, height, width)`, mientras que NumPy/PIL nos las da como `(batch, height, width, channels)`:

In [ ]:
import torch

X_img = torch.tensor(images).permute(0, 3, 1, 2)
y_img = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
X_img.shape

Partición 60/20/20, exactamente como en `NB12`, estratificando para que ambas clases sigan representadas en cada partición:

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(labels))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels)
idx_train, idx_val = train_test_split(
    idx_train_full, test_size=0.25, random_state=42, stratify=labels[idx_train_full]
)

X_train, y_train = X_img[idx_train], y_img[idx_train]
X_val, y_val = X_img[idx_val], y_img[idx_val]
X_test, y_test = X_img[idx_test], y_img[idx_test]

X_train.shape, X_val.shape, X_test.shape

---

## 8. Práctica: construir y entrenar un clasificador CNN

Tres bloques Conv+ReLU+Pool (16 → 32 → 64 filtros, cada vez más profundos a medida que el tamaño espacial se reduce — un patrón muy habitual), seguidos de una cabeza totalmente conectada regularizada con dropout, siguiendo directamente la práctica de `NB12`:

In [ ]:
import torch.nn as nn

class HullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 96 -> 48
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 48 -> 24
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 24 -> 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 12 * 12, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

torch.manual_seed(42)
cnn = HullCNN()
sum(p.numel() for p in cnn.parameters())

Merece la pena comparar ese número de parámetros con la estimación de la §2 de "1,7 millones de pesos para una sola capa de MLP" — los filtros compartidos de la convolución mantienen una red mucho más profunda y a la vez mucho más pequeña, incluso con tres capas convolucionales más una cabeza totalmente conectada.

Una breve nota sobre por qué esta forma concreta (16 → 32 → 64 filtros, tres pasos de pooling): sigue un patrón muy habitual y motivado empíricamente en arquitecturas CNN reales — a medida que el pooling reduce el tamaño espacial (96 → 48 → 24 → 12), *aumentamos* el número de filtros, de modo que cada capa mantiene aproximadamente el mismo "presupuesto" de información aunque cubra una vista más gruesa de la imagen. Las primeras capas, trabajando sobre la imagen a resolución completa, suelen necesitar solo unos pocos filtros para capturar patrones locales simples (bordes, transiciones de color); las capas más profundas, trabajando sobre una representación más gruesa pero semánticamente más rica, `se benefician de más filtros para representar una variedad más amplia de formas y texturas aprendidas`. Veremos un indicio de esto directamente en la Parte 9, cuando visualicemos qué detecta realmente la primerísima capa.

Entrena con la misma receta que `NB12`: Adam, entropía cruzada binaria, siguiendo la pérdida de entrenamiento y validación en cada época:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

n_epochs = 25
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    cnn.train()
    optimizer.zero_grad()
    outputs = cnn(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    cnn.eval()
    with torch.no_grad():
        val_loss = criterion(cnn(X_val), y_val)
    val_losses.append(val_loss.item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train loss: {loss.item():.4f} - val loss: {val_loss.item():.4f}")

Dibuja ambas curvas, exactamente como en `NB12`:

In [ ]:
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (binary cross-entropy)")
plt.title(f"CNN training — detecting '{target_class}'")
plt.legend()
plt.show()

**Pruébalo tú mismo**: la misma cuantificación que la Sección 4 de `NB12` — ¿qué época tuvo la menor pérdida de validación, y a qué distancia está de la última época entrenada?

In [ ]:
best_epoch = int(np.argmin(val_losses))
print(f"Best validation loss at epoch {best_epoch + 1} of {n_epochs}: {val_losses[best_epoch]:.4f}")
print(f"Final (epoch {n_epochs}) validation loss: {val_losses[-1]:.4f}")


---

## 9. Evaluar la CNN y visualizar qué ha aprendido

Evalúa sobre el conjunto de test intacto con las mismas herramientas que todo clasificador desde `NB08` — y, como esta es una tarea de inspección real, merece la pena leer la matriz de confusión pensando en las consecuencias operativas reales, no solo como cuatro números abstractos:

- Un **falso negativo** aquí (predecir "Ausente" cuando el crecimiento está realmente presente) significa que una sección del casco con incrustaciones pasa desapercibida — la condición sigue costando combustible y resistencia hasta que la siguiente inspección la detecte, o no.
- Un **falso positivo** (predecir "Presente" cuando el casco está realmente limpio) significa que un inspector dedica tiempo a revisar dos veces una sección que no lo necesitaba — esfuerzo perdido, pero un error mucho más barato que el falso negativo anterior.

Esta asimetría es exactamente el tipo de razonamiento que introdujo `NB07` con el ejemplo del vertido de petróleo — en un despliegue real probablemente ajustarías el umbral de clasificación (ahora mismo un `0.5` fijo en el código de abajo) para cambiar algunos falsos positivos por menos falsos negativos, en vez de tratar ambos tipos de error como igual de costosos:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cnn.eval()
with torch.no_grad():
    test_preds = (torch.sigmoid(cnn(X_test)) > 0.5).float()

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test.numpy().ravel()

# labels=[0, 1] keeps both classes in the report even if one happens to be
# briefly absent from this particular split or from the predictions -- with a
# small sample this is a real possibility, not just defensive styling.
print(confusion_matrix(y_test_np, y_pred_np, labels=[0, 1]))
print()
print(classification_report(
    y_test_np, y_pred_np, labels=[0, 1], target_names=["Absent", "Present"], zero_division=0
))

Las métricas resumen; no te muestran *qué* imágenes se están confundiendo, ni si los errores parecen razonables a un ojo humano. Mira un puñado de predicciones reales junto a las imágenes reales — para un modelo de visión, este tipo de comprobación cualitativa importa tanto como cualquier métrica aislada, y es práctica habitual antes de confiar en un modelo con imágenes de inspección reales:

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(8)):
    img_show = X_test[i].permute(1, 2, 0).numpy()
    ax.imshow(img_show)
    ax.set_title(f"True: {int(y_test_np[i])}  Pred: {int(y_pred_np[i])}")
    ax.axis("off")
plt.tight_layout()
plt.show()

Por último, mira *dentro* de la red: ¿a qué responde la primerísima capa convolucional en una imagen real de test? Esto es exactamente la idea de "características aprendidas automáticamente" de la §2, hecha visible — y una técnica habitual en la práctica real de visión por computador para comprobar que un modelo está mirando algo razonable, y no explotando algún artefacto no relacionado de las imágenes (un modo de fallo conocido: un modelo que "hace trampa" fijándose, por ejemplo, en una marca de agua de la cámara o en una diferencia de iluminación en vez de en la condición real que se está clasificando):

In [ ]:
sample_image = X_test[0].unsqueeze(0)
with torch.no_grad():
    first_layer_output = cnn.features[0](sample_image)  # after the first Conv2d, before ReLU/pool

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, ch in zip(axes.ravel(), range(8)):
    ax.imshow(first_layer_output[0, ch].numpy(), cmap="viridis")
    ax.set_title(f"Filter {ch}")
    ax.axis("off")
plt.suptitle("What the first conv layer's 8 filters respond to, on one real image")
plt.tight_layout()
plt.show()

**Lee tu propio resultado**: probablemente algunos filtros resaltan bordes o límites en la imagen, otros pueden responder a regiones de color/textura, y algunos pueden verse casi en blanco en esta imagen concreta — normal para una red sin ajustar, entrenada brevemente con una muestra pequeña. Nadie *diseñó* estos filtros; el detector de bordes de la §3 se construyó a mano para ilustrar el punto, pero cada filtro aquí se aprendió únicamente por descenso de gradiente minimizando la pérdida de clasificación.

Este patrón — las primeras capas detectando características simples y genéricas (bordes, colores, texturas) que se combinan en patrones cada vez más complejos y específicos de la tarea en las capas más profundas — se documentó rigurosamente en redes reales entrenadas (no en ejemplos de juguete) por [Zeiler y Fergus (2014), *Visualizing and Understanding Convolutional Networks*](https://arxiv.org/abs/1311.2901), uno de los artículos que convirtió la interpretabilidad de las CNN en un tema de investigación serio y no en una curiosidad. Es también la razón por la que el **transfer learning** (aprendizaje por transferencia) funciona: una clase posterior de este bloque reutilizará una red preentrenada con millones de fotos no relacionadas, bajo la suposición razonable de que `sus filtros tempranos y genéricos de bordes/texturas son útiles para casi cualquier tarea de imagen, incluida la nuestra`.

**Pruébalo tú mismo**: el texto anterior promete que las capas más profundas detectan "patrones cada vez más complejos y específicos de la tarea" — pero solo hemos mirado la *primera* capa. Visualiza la respuesta de la segunda capa convolucional en la misma imagen (`cnn.features[3]`, aplicada después del conv+ReLU+pool del primer bloque) y compara: ¿se ven estos mapas de características cualitativamente distintos de los de la primera capa?

In [ ]:
with torch.no_grad():
    after_first_block = cnn.features[0:3](sample_image)      # conv1 + ReLU + pool
    second_layer_output = cnn.features[3](after_first_block)  # conv2, before its own ReLU/pool

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, ch in zip(axes.ravel(), range(8)):
    ax.imshow(second_layer_output[0, ch].numpy(), cmap="viridis")
    ax.set_title(f"Filter {ch}")
    ax.axis("off")
plt.suptitle("What the second conv layer's first 8 filters respond to, on the same real image")
plt.tight_layout()
plt.show()


---

## Resumen de la clase

- Un MLP normal sobre píxeles en bruto tiene muchos parámetros e ignora la estructura espacial; la convolución arregla ambas cosas deslizando filtros pequeños, compartidos y aprendidos por la imagen.
- Un kernel/filtro produce un mapa de características; filtros distintos detectan patrones distintos, y la red aprende los valores de los filtros, no nosotros.
- El pooling reduce los mapas de características y añade tolerancia a pequeños desplazamientos; apilar bloques Conv+ReLU+Pool construye desde bordes hasta características cada vez más abstractas.
- Una vez extraídas las características, se aplica sin cambios la misma cabeza totalmente conectada, el dropout y la disciplina de entrenamiento de `NB11`/`NB12`.
- Construimos etiquetas de clasificación directamente a partir de un dataset real anotado a nivel de píxel (en vez de asumir que las etiquetas ya existen), entrenamos una CNN real, y miramos dentro de ella para ver qué había aprendido realmente.

## Para la próxima clase (NB14)

Cerramos el Bloque 3 con **modelos de secuencia** (RNN/LSTM) para datos dependientes del tiempo — un tipo de estructura distinto al de las imágenes, donde lo que la arquitectura necesita respetar es el *orden*, no la localidad espacial.

## Tarea / Ideas de práctica

1. Cambia `target_class` a una carpeta de máscara distinta (p. ej. corrosión, u otra clase de la lista impresa en la §5) y vuelve a entrenar — ¿funciona la CNN notablemente mejor o peor con esa condición? ¿Por qué podría ser (piensa en lo visualmente distintiva que es la condición, y en lo equilibradas que están las clases)?
2. Aumenta `sample_files` de 400 a 800 (o el dataset completo, si tienes tiempo y una GPU) — ¿más datos cierra la brecha entre la pérdida de entrenamiento y la de validación?
3. Añade un cuarto bloque Conv+ReLU+Pool (128 filtros) — recuerda recalcular el tamaño de entrada de la capa `Linear` para que coincida con el nuevo mapa de características, más pequeño.
4. Prueba `IMG_SIZE = 64` en vez de 96 — ¿cuánto más rápido va el entrenamiento, y cuánto (si algo) empeora la accuracy en test?
5. Explica con tus propias palabras por qué usamos `stratify=labels` en las particiones train/val/test aquí, relacionándolo con la comprobación de equilibrio de clases de la Parte 6.

> ***Como siempre: los números de un modelo importan, pero también importa mirar de verdad qué ha acertado y qué ha fallado — la cuadrícula de imágenes de la Parte 9 no es tarea opcional, es parte de evaluar cualquier modelo de visión real.***